# LIGHTGBM + DEFAULT PARAMS + SHAP LÀM BASELINE

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split, StratifiedKFold
import shap

In [ ]:
app_train_fe = pd.read_parquet("../Data/app_train_fe.parquet")
app_test_fe = pd.read_parquet("../Data/app_test_fe.parquet")

In [ ]:
# ==============================================================================
# BƯỚC 1: CHUẨN BỊ DỮ LIỆU
# ==============================================================================

# Tách features và target
X_train = app_train_fe.drop(columns=["TARGET"])
X_test = app_test_fe.copy()
y_train = app_train_fe["TARGET"]

# Loại bỏ SK_ID_CURR vì chỉ là ID định danh, không có giá trị dự báo
# Giữ lại để dùng sau (submit Kaggle, OOF predictions)
features = [col for col in X_train.columns if col not in ["SK_ID_CURR", "TARGET"]]
print(f"Số features đưa vào model: {len(features)}")

# Lấy danh sách cột category để khai báo với LightGBM
# LightGBM xử lý category trực tiếp mà không cần One-Hot hay Label Encoding
cat_cols = X_train[features].select_dtypes(include="category").columns.tolist()
print(f"Số cột category: {len(cat_cols)}")

# ==============================================================================
# BƯỚC 2: CROSS-VALIDATION 5 FOLD + THU THẬP SHAP
# ==============================================================================
# Dùng StratifiedKFold thay vì train_test_split đơn giản vì:
# - Dataset mất cân bằng (8% TARGET=1) → cần giữ nguyên tỷ lệ ở mỗi fold
# - 5 fold cho kết quả ổn định hơn 1 lần split ngẫu nhiên
# - OOF predictions có thể dùng cho stacking sau này
# - Thu thập SHAP từ tất cả 5 fold → 25,000 dòng đại diện hơn

params = {
    "objective": "binary",  # Bài toán phân loại nhị phân (vỡ nợ / không)
    "metric": "auc",  # Đánh giá bằng AUC — phù hợp với dataset mất cân bằng
    "learning_rate": 0.05,  # Learning rate nhỏ → học chậm nhưng ổn định hơn
    "num_leaves": 31,  # Số lá mỗi cây — baseline đơn giản, chưa tune
    "n_estimators": 1500,  # Số cây tối đa — early stopping sẽ dừng sớm nếu không cải thiện
    "random_state": 42,  # Seed để tái lập kết quả
    "n_jobs": -1,  # Dùng tất cả CPU available
    "subsample": 0.8,  # Lấy 80% dòng ngẫu nhiên mỗi cây → giảm overfitting
    "colsample_bytree": 0.8,  # Lấy 80% cột ngẫu nhiên mỗi cây → giảm overfitting
    "verbose": -1,  # Tắt log của LightGBM để output gọn hơn
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Mảng lưu dự đoán Out-of-Fold (OOF)
# OOF = dự đoán trên tập validation của mỗi fold, không bị data leakage
oof_preds = np.zeros(len(X_train))

# Danh sách lưu feature importance của từng fold để tính trung bình sau
feature_importance_list = []

# Danh sách lưu SHAP values từ tất cả 5 fold
# Mỗi fold lấy 5,000 dòng → tổng 25,000 dòng đại diện hơn 2,000 dòng từ 1 fold
all_shap_values = []
all_val_features = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train[features], y_train)):
    print(f"\nFold {fold+1}/5...")

    # Tách train/validation theo index của fold hiện tại
    X_tr = X_train[features].iloc[train_idx]
    X_val = X_train[features].iloc[val_idx]
    y_tr = y_train.iloc[train_idx]
    y_val = y_train.iloc[val_idx]

    # Khởi tạo và train model
    model = lgb.LGBMClassifier(**params)
    model.fit(
        X_tr,
        y_tr,
        eval_set=[(X_val, y_val)],
        # Early stopping: dừng nếu AUC validation không tăng sau 50 vòng
        # Tránh overfitting và tiết kiệm thời gian train
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
        categorical_feature=cat_cols,
    )

    # Dự đoán xác suất vỡ nợ trên tập validation
    # [:, 1] lấy xác suất của class 1 (vỡ nợ)
    oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]

    # Tính AUC và GINI cho fold này
    # GINI = 2*AUC - 1, phổ biến hơn trong credit scoring thực tế
    fold_auc = roc_auc_score(y_val, oof_preds[val_idx])
    fold_gini = fold_auc * 2 - 1
    print(
        f"Fold {fold+1} — AUC: {fold_auc:.4f} | GINI: {fold_gini:.4f} | Best iteration: {model.best_iteration_}"
    )

    # Lưu feature importance của fold này
    # feature_importances_ đo số lần mỗi feature được dùng để chia nhánh (split count)
    feature_importance_list.append(
        pd.DataFrame({"feature": features, "importance": model.feature_importances_})
    )

    # ===== THU THẬP SHAP VALUES =====
    # Lấy 5,000 dòng đại diện từ validation fold này
    # stratify=y_val: giữ nguyên tỷ lệ 8% nợ xấu trong mỗi sample
    # Dùng train_test_split chỉ để lấy mẫu, không dùng phần còn lại (_)
    X_val_sample, _, y_val_sample, _ = train_test_split(
        X_val, y_val, train_size=5000, stratify=y_val, random_state=42
    )

    # TreeExplainer được tối ưu riêng cho tree-based model như LightGBM
    # Nhanh hơn KernelExplainer nhiều lần nhờ tận dụng cấu trúc cây
    explainer = shap.TreeExplainer(model)

    # shap_values() trả về list 2 phần tử [class0, class1]
    # [1] để lấy SHAP của class 1 (vỡ nợ) — đây mới là phần quan tâm
    # Thay dòng này:
    shap_values_sample = explainer.shap_values(X_val_sample)[1]

    # Bằng đoạn này:
    shap_explanation = explainer(X_val_sample)

    # shap_explanation.values có shape (n_samples, n_features, n_classes)
    # hoặc (n_samples, n_features) tùy phiên bản
    print(f"shap_explanation.values shape: {shap_explanation.values.shape}")
    
    if shap_explanation.values.ndim == 3:
        # Binary classification: lấy class 1 (vỡ nợ)
        shap_values_sample = shap_explanation.values[:, :, 1]
    else:
        shap_values_sample = shap_explanation.values

    print(f"shap_value_sample shape: {shap_values_sample.shape}")
    # Gom kết quả từ fold này vào danh sách chung
    all_shap_values.append(shap_values_sample)
    all_val_features.append(X_val_sample)

    print(f"SHAP fold {fold+1} done — shape: {shap_values_sample.shape}")

# Tổng hợp kết quả OOF — đây là metric đáng tin cậy nhất
# vì mỗi dòng được dự đoán khi nó KHÔNG nằm trong tập train
oof_auc = roc_auc_score(y_train, oof_preds)
oof_gini = oof_auc * 2 - 1
print(f"\n{'='*50}")
print(f"OOF AUC:  {oof_auc:.4f}")
print(f"OOF GINI: {oof_gini:.4f}")

# ==============================================================================
# BƯỚC 3: FEATURE IMPORTANCE (SPLIT COUNT)
# ==============================================================================

# Tính trung bình importance qua 5 fold để kết quả ổn định hơn
# Dùng trung bình thay vì 1 fold bất kỳ vì mỗi fold có thể cho kết quả hơi khác nhau
feature_importance = (
    pd.concat(feature_importance_list)
    .groupby("feature")["importance"]
    .mean()
    .reset_index()
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

# Vẽ top 25 feature quan trọng nhất
plt.figure(figsize=(12, 9))
sns.barplot(
    x="importance", y="feature", data=feature_importance.head(25), palette="viridis"
)
plt.title(
    "Top 25 Feature Importance - LightGBM Baseline (avg 5 folds)",
    fontsize=14,
    fontweight="bold",
)
plt.xlabel("Importance (Split Count)")
plt.ylabel("Features")
plt.tight_layout()
plt.show()

# Kiểm tra vị trí các feature tự tạo trong bảng xếp hạng
# Để đánh giá xem feature engineering có thực sự đóng góp không
my_custom_features = [
    "EXT_SOURCE_MEAN",
    "EXT_SOURCE_PRODUCT",
    "EXT_SOURCE_STD",
    "DEBT_TO_INCOME",
    "ANNUITY_TO_INCOME",
    "CREDIT_TO_ANNUITY_MONTHS",
    "IMPLICIT_INTEREST_RATE",
    "TOTAL_DEBT_TO_INCOME",
    "BUREAU_CREDIT_UTILIZATION",
    "MAX_DPD_ALL_SOURCES",
    "TOTAL_LATE_MONTHS",
    "INSTALL_COMPLETION_RATE_NAN",
    "INSTALL_COMPLETION_RATE_ZERO",
    "EMPLOYMENT_LIFE_RATIO",
    "RESIDENCE_STABILITY",
    "RECENT_INQUIRY_SPIKE",
    "TOTAL_MISSING_FLAGS",
]
print("\nVị trí các feature tự tạo trong bảng Feature Importance:")
print(feature_importance[feature_importance["feature"].isin(my_custom_features)])

# ==============================================================================
# BƯỚC 4: SHAP VALUES (TỔNG HỢP TỪ 5 FOLD — 25,000 DÒNG)
# ==============================================================================

# Gộp SHAP values từ 5 fold lại thành 1 ma trận 25,000 dòng
# np.vstack: xếp chồng các ma trận theo chiều dọc
final_shap_matrix = np.vstack(all_shap_values)

# Gộp features matrix từ 5 fold lại
final_features_matrix = pd.concat(all_val_features, axis=0).reset_index(drop=True)

print(f"\nFinal SHAP matrix shape: {final_shap_matrix.shape}")  # (25000, 265)
print(f"Final features matrix shape: {final_features_matrix.shape}")

# Tính mean |SHAP| cho từng feature
# Đây là chỉ số đáng tin cậy nhất để lọc feature:
# - Không bị bias bởi cột có nhiều giá trị unique (như split count)
# - Phản ánh đóng góp thực tế của feature vào dự đoán
mean_shap = (
    pd.DataFrame(
        {"feature": features, "mean_abs_shap": np.abs(final_shap_matrix).mean(axis=0)}
    )
    .sort_values("mean_abs_shap", ascending=False)
    .reset_index(drop=True)
)

print("Xuất file mean_shap ra để dùng tiếp: ")
mean_shap.to_csv("../Data/mean_shap.csv")
print("\nTop 30 features theo Mean |SHAP|:")
print(mean_shap.head(30))

# Số features có SHAP = 0 (hoàn toàn không đóng góp)
zero_shap = mean_shap[mean_shap["mean_abs_shap"] == 0]
print(f"\nSố features có SHAP = 0: {len(zero_shap)}")
print(zero_shap["feature"].tolist())

# SHAP Summary Plot từ 25,000 dòng
# Mỗi chấm = 1 dòng dữ liệu
# Trục X = SHAP value (âm = giảm rủi ro, dương = tăng rủi ro)
# Màu = giá trị feature (đỏ = cao, xanh = thấp)
plt.figure(figsize=(12, 10))
shap.summary_plot(final_shap_matrix, final_features_matrix, max_display=25, show=False)
plt.title(
    "SHAP Beeswarm Plot - Top 25 Features (25,000 dòng từ 5 folds)",
    fontsize=14,
    fontweight="bold",
)
plt.tight_layout()
plt.show()

# SHAP Bar Plot — trung bình |SHAP| theo từng feature
# Ít bị bias hơn split count, dùng để quyết định loại feature nào
plt.figure(figsize=(12, 8))
shap.summary_plot(
    final_shap_matrix,
    final_features_matrix,
    plot_type="bar",
    max_display=25,
    show=False,
)
plt.title(
    "SHAP Feature Importance (Mean |SHAP Value|) — 25,000 dòng từ 5 folds",
    fontsize=14,
    fontweight="bold",
)
plt.tight_layout()
plt.show()

# ==============================================================================
# BƯỚC 5: LƯU OOF PREDICTIONS (CHO STACKING SAU NÀY)
# ==============================================================================

# OOF predictions dùng để:
# 1. Stacking: làm input cho meta-model
# 2. Phân tích lỗi: xem model sai ở đâu
# 3. Calibration: hiệu chỉnh xác suất nếu cần
oof_df = pd.DataFrame(
    {
        "SK_ID_CURR": X_train["SK_ID_CURR"],
        "TARGET": y_train.values,
        "OOF_PRED": oof_preds,
    }
)
print(f"\nOOF predictions đã lưu: {oof_df.shape}")
print(oof_df.head())